# HAJIMI 系统集成测试

> **维护人**：涂浚稷 (20230353)  
> **日期**：2026-07-03  
> **用途**：一键验证所有组件状态，按 `Shift+Enter` 逐格执行

## 一、环境检查

In [ ]:
# 项目根目录
import os, sys
PROJECT = r"E:\Fuzzy-Visual-Assisted-Question-Answering-System"
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)
print(f"项目目录: {PROJECT}")
print(f"Python:   {sys.version.split()[0]}")

In [ ]:
# 检查关键依赖
deps = {
    "fastapi": "A端", "uvicorn": "A端", "sqlalchemy": "A端",
    "httpx": "HTTP", "psutil": "监控",
    "speech_recognition": "ASR", "pyttsx3": "TTS", "vosk": "Vosk"
}
for mod, role in deps.items():
    try:
        __import__(mod)
        print(f"  ✅ {mod:25s} ({role})")
    except ImportError:
        print(f"  ❌ {mod:25s} ({role}) — pip install {mod}")

## 二、端口检测

检查 HAJIMI 使用的端口是否被占用或空闲。

In [ ]:
import subprocess

ports = {
    8000: "A端 Server (独立)",
    8002: "OmniParser (D:\\ominprester)",
    8010: "B端嵌入式 A端",
    5173: "Web 管理面板 (Vite)",
}

print("端口占用状态:\n")
for port, desc in ports.items():
    result = subprocess.run(
        f'cmd //c "netstat -ano | findstr :{port} | findstr LISTENING"',
        capture_output=True, text=True, shell=True
    )
    if result.stdout.strip():
        pid = result.stdout.strip().split()[-1]
        print(f"  🟢 :{port} ({desc})")
        print(f"      PID: {pid}")
    else:
        print(f"  ⚪ :{port} ({desc}) — 空闲")

## 三、A 端 Server 启动

启动 A 端并验证 health 端点。如果已在运行则跳过。

In [ ]:
import httpx, time, threading

SERVER = "http://127.0.0.1:8000"

# 检查是否已在运行
try:
    r = httpx.get(f"{SERVER}/api/demo/health", timeout=2)
    if r.status_code == 200:
        print(f"✅ A端已在运行: {r.json()}")
    else:
        print(f"⚠ A端返回 {r.status_code}")
except Exception:
    print("A端未启动 — 正在启动...")
    # 后台启动
    import subprocess
    proc = subprocess.Popen(
        ["python", "-m", "uvicorn", "server.main:app",
         "--host", "127.0.0.1", "--port", "8000"],
        cwd=PROJECT, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    print(f"  启动中 (PID {proc.pid})，等待就绪...")
    for i in range(15):
        time.sleep(2)
        try:
            r = httpx.get(f"{SERVER}/api/demo/health", timeout=2)
            if r.status_code == 200:
                print(f"  ✅ 就绪! (耗时 ~{(i+1)*2}s)")
                break
        except Exception:
            pass
    else:
        print("  ❌ 启动超时，请手动检查")

## 四、A 端 26 端点全量检测

In [ ]:
import httpx

ENDPOINTS = [
    # (方法, 路径, 认证头, 请求体)
    ("GET",  "/api/demo/health",              None, None),
    ("POST", "/api/audit/report",              "X-Demo-Key",
        {"client_id":"nb","batch":[{"task_id":"nb-001","query":"test","intent_category":"operation_guide","route":"L2","total_steps":1,"completed_steps":1,"result":"success","duration_ms":100,"fingerprint_mismatches":0,"redline_triggered":False}]}),
    ("POST", "/api/audit/feedback",            "X-Demo-Key",
        {"task_id":"nb-001","feedback_type":"useful","comment":"测试"}),
    ("GET",  "/api/config/pull",               "X-Demo-Key", None),
    ("POST", "/api/auth/login",                 None,
        {"username":"admin@hajimi.local","password":"test123"}),
    ("GET",  "/api/admin/stats/overview",      "X-Admin-Key", None),
    ("GET",  "/api/admin/stats/top-tasks",     "X-Admin-Key", None),
    ("GET",  "/api/admin/stats/trend",         "X-Admin-Key", None),
    ("GET",  "/api/admin/stats/redline",       "X-Admin-Key", None),
    ("GET",  "/api/admin/stats/feedback",      "X-Admin-Key", None),
    ("GET",  "/api/admin/failures/list",       "X-Admin-Key", None),
    ("GET",  "/api/admin/config/current",      "X-Admin-Key", None),
    ("GET",  "/api/admin/flow/topology",       "X-Admin-Key", None),
    ("GET",  "/api/admin/flow/metrics",        "X-Admin-Key", None),
    ("GET",  "/api/admin/flow/versions",       "X-Admin-Key", None),
    ("GET",  "/api/admin/monitor/health",      "X-Admin-Key", None),
    ("GET",  "/api/admin/monitor/alerts",      "X-Admin-Key", None),
    ("POST", "/api/admin/monitor/alerts/read-all","X-Admin-Key", None),
]

passed, failed = 0, 0
for method, path, auth, body in ENDPOINTS:
    headers = {}
    if auth:
        headers[auth] = "hajimi-demo-2026"
    try:
        if method == "GET":
            r = httpx.get(f"{SERVER}{path}", headers=headers, timeout=5)
        else:
            r = httpx.post(f"{SERVER}{path}", headers=headers, json=body, timeout=5)
        if r.status_code == 200:
            print(f"  ✅ {method:4s} {path}")
            passed += 1
        else:
            print(f"  ⚠ {method:4s} {path} → {r.status_code}")
            failed += 1
    except Exception as e:
        print(f"  ❌ {method:4s} {path} → {e}")
        failed += 1

print(f"\n  结果: {passed} 通过, {failed} 失败")

## 五、C 端模块导入检查

In [ ]:
modules = [
    "client.voice.asr_client",
    "client.voice.tts_engine",
    "client.audit.audit_agent",
    "client.config.config_poller",
    "client.integration.controller",
    "client.bc_adapter",
]
for m in modules:
    try:
        __import__(m)
        print(f"  ✅ {m}")
    except Exception as e:
        print(f"  ❌ {m}: {e}")

## 六、C 端全量测试

依次运行 Day 2-4 检查脚本、B-C 集成仿真、审计 E2E 测试。

In [ ]:
import subprocess, sys

tests = [
    ("Day 2 检查",    "client/day2_check.py"),
    ("Day 3 检查",    "client/day3_check.py"),
    ("Day 4 检查",    "client/day4_check.py"),
    ("B-C 集成仿真",  "client/bc_integration_test.py"),
    ("审计 E2E",      "client/audit_e2e_test.py"),
]

for name, script in tests:
    print(f"\n{'─'*50}\n  {name}\
{'─'*50}")
    result = subprocess.run(
        [sys.executable, script], cwd=PROJECT,
        capture_output=True, text=True, timeout=60
    )
    # 只打印最后 5 行（结果行）
    lines = result.stdout.strip().split("\n")
    for line in lines[-5:]:
        print(f"  {line}")
    if result.returncode != 0:
        print(f"  ⚠ 退出码 {result.returncode}")
        print(result.stderr[-300:])

## 七、Web 管理面板启动

In [ ]:
import os

web_dir = os.path.join(PROJECT, "web-admin")
node_modules = os.path.join(web_dir, "node_modules")

if not os.path.isdir(node_modules):
    print("node_modules 不存在 — 正在 npm install...")
    subprocess.run(["npm", "install"], cwd=web_dir, shell=True)
    print("安装完成")
else:
    print("✅ node_modules 已就绪")

print(f"\n启动 Web 管理面板:\n")
print(f"  终端: cd web-admin && npm run dev")
print(f"  浏览器打开: http://localhost:5173")
print(f"  登录: 任意密码")

## 八、端口清理（如需要）

杀掉 A 端、OmniParser 的占用进程。

In [ ]:
import subprocess

print("⚠ 此操作会强制终止 A 端和 OmniParser 进程！")
confirm = input("输入 'yes' 确认: ")

if confirm.lower() == "yes":
    for port in [8000, 8002, 8010, 5173]:
        result = subprocess.run(
            f'cmd //c "netstat -ano | findstr :{port} | findstr LISTENING"',
            capture_output=True, text=True, shell=True
        )
        if result.stdout.strip():
            pid = result.stdout.strip().split()[-1]
            subprocess.run(f"cmd //c \"taskkill /F /PID {pid}\"", shell=True)
            print(f"  🗑 已杀 :{port} (PID {pid})")
        else:
            print(f"  ⚪ :{port} 空闲")
    print("端口清理完成")
else:
    print("已取消")

## 九、一键全测汇总

以上全部步骤等效于终端命令：

```bash
# 启动 A 端
python -m uvicorn server.main:app --host 127.0.0.1 --port 8000

# 全量测试
python client/day2_check.py
python client/day3_check.py
python client/day4_check.py
python client/bc_integration_test.py
python client/audit_e2e_test.py

# Web 面板
cd web-admin && npm run dev

# 端口清理
cmd //c "for /f \"tokens=5\" %a in ('netstat -ano ^| findstr /R \":8000 :8002 :8010\" ^| findstr LISTENING') do taskkill /F /PID %a"
```

---

**常见问题**：

| 问题 | 解决 |
|------|------|
| `pip install` 失败 | `pip install --user <package>` |
| 端口被占用 | 执行"端口清理"格 |
| A 端启动慢 | 正常，V1 初始化 7 个数据库表需 5-8s |
| 测试全绿 | ✅ C 端一切正常！ |